# 01. 定义Jit装饰的算子

In [1]:
import tilelang
import tilelang.ascend.language as T

@tilelang.jit(out_idx=[-1])
def my_add():

    @T.prim_func
    def main(
        A: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
        B: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
        C: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
    ):
        
        with T.Kernel(1) as _:
            a_ub = T.alloc_shared(A.shape, A.dtype)
            b_ub = T.alloc_shared(B.shape, B.dtype)
            c_ub = T.alloc_shared(C.shape, C.dtype)

            T.copy(A[0, 0], a_ub)
            T.copy(B[0, 0], b_ub)

            c_ub[0, 0] = a_ub[0, 0] + b_ub[0, 0]

            T.copy(c_ub, C[0, 0])

    return main

my_jit_add = my_add()
print(my_jit_add.get_kernel_source())


Loading tilelang libs from dev root: /data/caojian/workspace/git/github/tile-ai/tilelang/build
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to prevent some unknown errors
  warnings.warn(


#include <tl_templates/ascend/common.h>
#include <tl_templates/ascend/debug.h>
extern "C" __global__ __vector__ void main_kernel(__gm__ float* A, __gm__ float* B, __gm__ float* C) {
  asc_init();
  __ubuf__ uint8_t *buf_dyn_shmem = (__ubuf__ uint8_t *)0;
  asc_copy_gm2ub_align((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[0]))), (__gm__ uint8_t*)((&(A[0]))), 1, 4, 0, 0, 0, static_cast<asc_load_l2_cache_mode>(0), 4, 4);
  asc_copy_gm2ub_align((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[8]))), (__gm__ uint8_t*)((&(B[0]))), 1, 4, 0, 0, 0, static_cast<asc_load_l2_cache_mode>(0), 4, 4);
  asc_sync_notify(PIPE_MTE2, PIPE_S, static_cast<event_t>(0));
  asc_sync_wait(PIPE_MTE2, PIPE_S, static_cast<event_t>(0));
  ((__ubuf__ float*)buf_dyn_shmem)[16] = (((__ubuf__ float*)buf_dyn_shmem)[0] + ((__ubuf__ float*)buf_dyn_shmem)[8]);
  asc_sync_notify(PIPE_S, PIPE_MTE3, static_cast<event_t>(0));
  asc_sync_wait(PIPE_S, PIPE_MTE3, static_cast<event_t>(0));
  asc_copy_ub2gm_align(

# 02. 调用算子&评测

In [2]:
import torch

torch.manual_seed(0)

a = torch.rand((1, 1), dtype=torch.float32).npu()
b = torch.rand((1, 1), dtype=torch.float32).npu()
c_golden = a + b

print(f"Input tensor a: {a}")
print(f"Input tensor b: {b}")
print(f"Golden tensor c: {c_golden}")

c = my_jit_add(a, b)
print(f"Output tensor c: {c}")

torch.testing.assert_close(c, c_golden, rtol=1e-5, atol=1e-5)
print("Test passed: The output tensor c is equal to the sum of the input tensors a and b.")

Input tensor a: tensor([[0.4963]], device='npu:0')
Input tensor b: tensor([[0.7682]], device='npu:0')
Golden tensor c: tensor([[1.2645]], device='npu:0')
Output tensor c: tensor([[1.2645]], device='npu:0')
Test passed: The output tensor c is equal to the sum of the input tensors a and b.


[W1013 12:44:47.709258785 ToKernelNpu.cpp:40] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())
